In [58]:
import os
import json
from game import sample, sample_until, sample_multi
from tqdm import tqdm
import random
import numpy as np
from xgboost import XGBClassifier
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss, mean_squared_error
from sklearn.calibration import CalibratedClassifierCV, FrozenEstimator
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

In [2]:
ranks = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER']
tiers = ['IV', 'III', 'II', 'I']

In [3]:
with open('champion_to_index.json') as f:
    champion_to_idx = json.load(f)

In [4]:
def rank_to_mmr(rank, tier):
    if rank == 'MASTER':
        tier = 'IV'
    return ranks.index(rank) * 4 + tiers.index(tier)

In [5]:
def vectorize(sample):
    arr = []

    dragon_names = ['WATER_DRAGON', 'AIR_DRAGON', 'CHEMTECH_DRAGON', 'FIRE_DRAGON', 'HEXTECH_DRAGON', 'EARTH_DRAGON']

    champion_vector = [0] * 172

    for champion in sample['champions'][:5]:
        champion_vector[champion_to_idx[champion]] = 1

    for champion in sample['champions'][5:]:
        champion_vector[champion_to_idx[champion]] = -1

    arr += champion_vector

    for team in sample['teams']:
        for player in team['players']:
            arr.append(player['kill_gold'])
            arr.append(player['baron_timer'])
            arr.append(player['elder_timer'])
            arr.append(player['death_timer'])
            arr.append(player['level'])
            
        for i in range(4):
            one_hot = [0, 0, 0, 0, 0, 0]
            if i < len(team['drakes']):
                name = team['drakes'][i]
                one_hot[dragon_names.index(name)] = 1
            arr += one_hot
        arr.append(team['rifts'])
        arr.append(team['atakhan'])
        arr.append(team['grubs'])
        arr += team['towers']
        arr += team['inhibs']
    arr.append(sample['time'])
    arr.append(int(sample['win']))

    return arr

In [6]:
def map_dataset(callback, split='train'):
    responses = []

    for rank in ranks:
        if rank != 'MASTER':
            for tier in tiers:
                for file in os.listdir(f'../dataset/{split}/{rank}/{tier}'):
                    path = f'../dataset/{split}/{rank}/{tier}' + f'/{file}'
                    r = callback(path, rank, tier)
                    if r:
                        responses.append(r)
        else:
            for file in os.listdir(f'../dataset/{split}/{rank}'):
                path = f'../dataset/{split}/{rank}' + f'/{file}'
                r = callback(path, rank, None)
                if r:
                    responses.append(r)

    return responses

In [13]:
def get_game_path(path, rank, tier):
    return path, rank, tier

all_game_paths = map_dataset(get_game_path, split='train')

In [14]:
len(all_game_paths)

86793

In [15]:
random.shuffle(all_game_paths)

In [16]:
num_train_games = int(len(all_game_paths) * 0.9)

In [22]:
def load_dataset(games):
    samples = []
    labels = []

    for path, rank, tier in games:
        with open(path) as f:
            game_data = json.load(f)
            if len(game_data['events']) > 30:
                indices = random.sample(range(0, len(game_data['events'])), 10)
                states = sample_multi(game_data, indices)
                for s in states:
                    v = vectorize(s)
                    inputs = v[:-1]
                    label = v[-1]
                    inputs.append(rank_to_mmr(rank, tier))
                    samples.append(inputs)
                    labels.append(label)

    return np.array(samples), np.array(labels)

In [23]:
x_train, y_train = load_dataset(all_game_paths[:num_train_games])
x_val, y_val = load_dataset(all_game_paths[num_train_games:])

In [52]:
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)

In [102]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(hidden_layer_sizes=(8), max_iter=1, warm_start=True))
])

mlp = pipe.named_steps["mlp"]

In [103]:
best_val_loss = float("inf")
patience = 5
no_improve = 0

for epoch in range(20):
    pipe.fit(x_train, y_train)
    val_pred = pipe.predict(x_val)
    val_loss = mean_squared_error(y_val, val_pred)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = mlp.coefs_, mlp.intercepts_
        no_improve = 0
        print(f'Found new best loss: {val_loss}')
    else:
        no_improve += 1
    
    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        mlp.coefs_, mlp.intercepts_ = best_weights
        break

C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Found new best loss: 0.17227325641375937


C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Found new best loss: 0.16969433407896042


C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


...

C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Found new best loss: 0.16814235928085514


C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization

Early stopping at epoch 12


C:\Users\fahd3\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


In [104]:
pred = pipe.predict(x_val)

In [108]:
pred

array([0.56159278, 0.51564099, 0.439796  , ..., 0.90235073, 0.96457998,
       0.90434228], shape=(85870,))

In [106]:
brier_score_loss(y_val, pred)

ValueError: y_proba contains values greater than 1.

In [96]:
(pred.round() == y_val).sum() / len(y_val)

np.float64(0.6533131477815303)

In [97]:
((pred >= 0.5) == y_val).sum() / len(y_val)

np.float64(0.6600442529404914)

In [40]:
pipe.score(x_train, y_train)

0.9905244207281535

In [7]:
samples = []
labels = []

def unpack_game(path, rank, tier):
    with open(path) as f:
        game_data = json.load(f)
        i = random.randint(0, len(game_data['events'])-1)
        s = sample(game_data, i)
        v = vectorize(s)
        inputs = v[:-1]
        label = v[-1]
        inputs.append(rank_to_mmr(rank, tier))
        samples.append(inputs)
        labels.append(label)

map_dataset(unpack_game, split='test')

[]

In [8]:
X = np.array(samples)
y = np.array(labels)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05
)

In [11]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [12]:
model.save_model("winprob_model.json")

In [13]:
probs = model.predict_proba(X_test)[:, 1]

In [14]:
auc = roc_auc_score(y_test, probs)
brier = brier_score_loss(y_test, probs)
ll = log_loss(y_test, probs)

print("AUC:   ", auc)
print("Brier: ", brier)
print("Logloss:", ll)

AUC:    0.8191615478252565
Brier:  0.1725618285080238
Logloss: 0.5096316231702733


In [30]:
def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bins[i], bins[i+1]
        mask = (probs > left) & (probs <= right)
        if np.sum(mask) > 0:
            bin_confidence = np.mean(probs[mask])
            bin_accuracy = np.mean(labels[mask])
            ece += (np.sum(mask) / len(probs)) * abs(bin_confidence - bin_accuracy)
    return ece

In [31]:
ece = compute_ece(probs, y_test)
print("ECE:", ece * 100)

NameError: name 'y_test' is not defined

In [17]:
calibrated = CalibratedClassifierCV(FrozenEstimator(model), method='isotonic')

In [18]:
calibrated.fit(X_test, y_test)

CalibratedClassifierCV(estimator=FrozenEstimator(estimator=XGBClassifier(base_score=None,
                                                                         booster=None,
                                                                         callbacks=None,
                                                                         colsample_bylevel=None,
                                                                         colsample_bynode=None,
                                                                         colsample_bytree=None,
                                                                         device=None,
                                                                         early_stopping_rounds=None,
                                                                         enable_categorical=False,
                                                                         eval_metric='logloss',
                                                                         feature_types=None,
                                                                         feature_weights=None,
                                                                         gamma=None,
                                                                         grow_policy=None,
                                                                         importance_type=None,
                                                                         interaction_constraints=None,
                                                                         learning_rate=0.05,
                                                                         max_bin=None,
                                                                         max_cat_threshold=None,
                                                                         max_cat_to_onehot=None,
                                                                         max_delta_step=None,
                                                                         max_depth=6,
                                                                         max_leaves=None,
                                                                         min_child_weight=None,
                                                                         missing=nan,
                                                                         monotone_constraints=None,
                                                                         multi_strategy=None,
                                                                         n_estimators=500,
                                                                         n_jobs=None,
                                                                         num_parallel_tree=None, ...)),
                       method='isotonic')

In [12]:
probs = calibrated.predict_proba(X)[:, 1]

In [15]:
ece = compute_ece(probs, y)
print("ECE:", ece * 100)

ECE: 1.0063306014791622


In [ ]:
# Save
joblib.dump(calibrated, "calibrated_model.pkl")

In [38]:
calibrated = joblib.load("calibrated_model.pkl")

In [20]:
def event_detector(state, event):
    if event['type'] == 'ITEM_PURCHASED' and event['itemId'] == 3031:
        return True, (event['participantId'] - 1) // 5
    return False, None

In [21]:
def callback(path, rank, tier):
    with open(path) as f:
        game_data = json.load(f)
        state, teamId = sample_until(game_data, event_detector)
        if state:
            v = vectorize(state)
            inputs = v[:-1]
            label = v[-1]
            inputs.append(rank_to_mmr(rank, tier))
            return (inputs, label, teamId)

states = map_dataset(callback, split='test')

In [22]:
inputs = []
labels = []
team_ids = []

for a in states:
    inputs.append(a[0])
    labels.append(a[1])
    team_ids.append(a[2])

In [23]:
X = np.array(inputs)

In [24]:
probs = calibrated.predict_proba(X)[:, 1]

In [25]:
for i, team in enumerate(team_ids):
    if team == 1:
        labels[i] = 1 - labels[i]
        probs[i] = 1 - probs[i]

In [26]:
y = np.array(labels)

In [27]:
win_probability_before = probs.mean()

In [28]:
win_rate = y.mean()

In [29]:
wpe = round(((win_rate - win_probability_before) * 100).item(), 2)
wpb = round((win_probability_before * 100).item(), 2)
wr = round((win_rate * 100).item(), 2)

print(f'+{wpe}% ({wr} - {wpb}) | {len(y)}')
print(f'mapped: +{round(map_value(wpe, 0, 100 - wpb, 0, 100), 2)}%')

+0.75% (57.53 - 56.78) | 71377


NameError: name 'map_value' is not defined